# Final Machine Learning Pipeline

Este notebook crea el PIPELINE END-TO-END definitivo para producción. **Características:**

- Usa exclusivamente librerías open source: `sklearn` y `feature-engine`.
- Incluye transformadores personalizados, replicando toda la lógica de feature engineering.
- SOLO usa las features seleccionadas tras el proceso de selección (notebook 04).
- El modelo entrenado (`Lasso` o `LogisticRegression`) es el paso final del pipeline.
- El pipeline puede procesar datos crudos y devolver predicciones directas (`predict`).
- El pipeline es persistible y autocontenible (`joblib.dump`/`joblib.load`).

---

## 1. Imports requeridos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, Binarizer
from sklearn.base import BaseEstimator, TransformerMixin

from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.missing_data import AddMissingIndicator
from feature_engine.transformation import LogTransformer, YeoJohnsonTransformer
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
from feature_engine.selection import DropFeatures
from feature_engine.wrappers import SklearnTransformerWrapper

---

## 2. Cargar dataset y train/test split

In [ ]:
# Cargar datos (reemplaza nombre de archivo según corresponda)
data = pd.read_csv('data/train.csv')

# Asegurarse de castear variables categóricas disfrazadas
data['MSSubClass'] = data['MSSubClass'].astype(str)  # ejemplo

# train/test split
X = data.drop(['SalePrice'], axis=1)
y = data['SalePrice'].copy()

# Si es regresión de housing, usar log-transform en target
y = np.log(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

---

## 3. Configuración: Constantes de features y mapeos
(Sólo se listan constantes cuyos elementos aparecen en FEATURES)

In [ ]:
# Cargar lista de features seleccionadas
tmp = pd.read_csv('selected_features.csv')
FEATURES = tmp['feature'].tolist()

# Ejemplo de definición — personaliza las siguientes según tu selección
CATEGORICAL_VARS_WITH_NA_MISSING = [
    'Alley', 'GarageType',
] if 'Alley' in FEATURES else []

CATEGORICAL_VARS_WITH_NA_FREQUENT = [
    'KitchenQual',
] if 'KitchenQual' in FEATURES else []

NUMERICAL_VARS_WITH_NA = [
    'LotFrontage',
] if 'LotFrontage' in FEATURES else []

TEMPORAL_VARS = ['YearBuilt'] if 'YearBuilt' in FEATURES else []
REF_VAR = 'YrSold' if 'YrSold' in FEATURES else None
DROP_FEATURES = ['Id'] if 'Id' in FEATURES else []

NUMERICALS_LOG_VARS = ['GrLivArea'] if 'GrLivArea' in FEATURES else []
NUMERICALS_YEO_VARS = ['TotalBsmtSF'] if 'TotalBsmtSF' in FEATURES else []
BINARIZE_VARS = ['CentralAir'] if 'CentralAir' in FEATURES else []

QUAL_VARS = ['ExterQual'] if 'ExterQual' in FEATURES else []
QUAL_MAPPINGS = {'ExterQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1}} if 'ExterQual' in FEATURES else {}
CATEGORICAL_VARS = [var for var in X.columns if X[var].dtype == 'O' and var in FEATURES]

---

## 4. Reducir datasets a FEATURES seleccionadas

In [ ]:
X_train = X_train[FEATURES].copy()
X_test = X_test[FEATURES].copy()

# Asegurarse de que las categóricas sean str
def ensure_object(X, cat_vars):
    for var in cat_vars:
        X[var] = X[var].astype(str)
    return X

X_train = ensure_object(X_train, CATEGORICAL_VARS)
X_test = ensure_object(X_test, CATEGORICAL_VARS)

---

## 5. Custom Transformers (si aplican)

In [ ]:
class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, variables, ref_variable):
        self.variables = variables
        self.ref_variable = ref_variable
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[self.ref_variable] - X[var]
        return X

class Mapper(BaseEstimator, TransformerMixin):
    def __init__(self, mapping):
        self.mapping = mapping
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for feature, mapper in self.mapping.items():
            X[feature] = X[feature].map(mapper).astype(float)
        return X

---

## 6. Pipeline end-to-end con modelo incluido

In [ ]:
steps = []

# Temporal variables
if TEMPORAL_VARS and REF_VAR:
    steps.append(('temporal', TemporalVariableTransformer(TEMPORAL_VARS, REF_VAR)))

# Imputación numérica
if NUMERICAL_VARS_WITH_NA:
    steps.append(('num_imputer', MeanMedianImputer(imputation_method='median', variables=NUMERICAL_VARS_WITH_NA)))

# Imputación categórica NA a 'Missing'
if CATEGORICAL_VARS_WITH_NA_MISSING:
    steps.append(('cat_na_missing', CategoricalImputer(imputation_method='missing', variables=CATEGORICAL_VARS_WITH_NA_MISSING)))

# Imputación categórica NA a más frecuente
if CATEGORICAL_VARS_WITH_NA_FREQUENT:
    steps.append(('cat_na_freq', CategoricalImputer(imputation_method='frequent', variables=CATEGORICAL_VARS_WITH_NA_FREQUENT)))

# Indicadores de missing
if NUMERICAL_VARS_WITH_NA:
    steps.append(('missing_ind', AddMissingIndicator(variables=NUMERICAL_VARS_WITH_NA)))

# Log transform numéricas (positivas)
if NUMERICALS_LOG_VARS:
    steps.append(('log_transform', LogTransformer(variables=NUMERICALS_LOG_VARS)))

# Yeo-Johnson transform numéricas (positivas o negativas)
if NUMERICALS_YEO_VARS:
    steps.append(('yeo_transform', YeoJohnsonTransformer(variables=NUMERICALS_YEO_VARS)))

# Binarización de variables
if BINARIZE_VARS:
    steps.append(('binarize', SklearnTransformerWrapper(Binarizer(), variables=BINARIZE_VARS)))

# Label encoding de variables cualitativas con mapeos custom
if QUAL_VARS and QUAL_MAPPINGS:
    steps.append(('qual_map', Mapper(QUAL_MAPPINGS)))

# Codificación de clases raras (depende del dominio)
if CATEGORICAL_VARS:
    steps.append(('rare_label', RareLabelEncoder(tol=0.01, n_categories=1, variables=CATEGORICAL_VARS)))
    steps.append(('ordinal', OrdinalEncoder(encoding_method='arbitrary', variables=CATEGORICAL_VARS)))

# Eliminar features innecesarios
if DROP_FEATURES:
    steps.append(('drop_features', DropFeatures(features_to_drop=DROP_FEATURES)))

# Escalado final
steps.append(('scaler', MinMaxScaler()))

# Agregar el modelo entrenado como último paso
steps.append(('lasso', Lasso(alpha=0.001, random_state=0)))

# Construcción del Pipeline final
total_pipe = Pipeline(steps)

---

## 7. Entrenamiento del pipeline

In [ ]:
price_pipe = total_pipe.fit(X_train, y_train)

---

## 8. Evaluación del pipeline

In [ ]:
# Train
y_pred_train = price_pipe.predict(X_train)
# Test
y_pred_test = price_pipe.predict(X_test)

# Invertir log-transform del target
y_train_orig = np.exp(y_train)
y_test_orig = np.exp(y_test)
y_pred_train_orig = np.exp(y_pred_train)
y_pred_test_orig = np.exp(y_pred_test)

print(f"Train MSE: {mean_squared_error(y_train_orig, y_pred_train_orig):.2f}")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train_orig, y_pred_train_orig)):.2f}")
print(f"Train R2: {r2_score(y_train_orig, y_pred_train_orig):.3f}")
print(f"Test MSE: {mean_squared_error(y_test_orig, y_pred_test_orig):.2f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_orig, y_pred_test_orig)):.2f}")
print(f"Test R2: {r2_score(y_test_orig, y_pred_test_orig):.3f}")

Las métricas obtenidas deben coincidir (±0.01) con las evaluadas en el entrenamiento manual del notebook 06. Esto valida que tu pipeline DEFINITIVO replica fielmente el modelo productivo.

### Scatter plot: predicciones vs valores reales

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test_orig, y_pred_test_orig, alpha=0.5)
plt.plot([y_test_orig.min(), y_test_orig.max()], [y_test_orig.min(), y_test_orig.max()], 'r--')
plt.xlabel('Real')
plt.ylabel('Predicción')
plt.title('Predicciones vs Real (test)')
plt.show()

### Histograma de errores de predicción

In [ ]:
errors = y_test_orig - y_pred_test_orig
plt.figure(figsize=(6,3))
plt.hist(errors, bins=30, edgecolor='k', alpha=0.7)
plt.title('Histograma de errores en test set')
plt.xlabel('Error')
plt.show()

---

## 9. Guardar el pipeline completo

In [ ]:
joblib.dump(price_pipe, 'proyecto_pipeline_machine_learning_muestra/pipeline_completo.joblib')
print('Pipeline end-to-end guardado exitosamente en pipeline_completo.joblib')

---

## 10. Scoring de datos nuevos

Este paso simula el uso real: datos nuevos pueden tener features no vistas, NaN inesperados, etc.

In [ ]:
# Cargar nuevos datos para scoring
data_new = pd.read_csv('data/test.csv')  # O cualquier otro dataset futuro

if 'MSSubClass' in data_new.columns:
    data_new['MSSubClass'] = data_new['MSSubClass'].astype(str)

# Reducir a sólo las features seleccionadas
data_new = data_new[FEATURES]

# Asegurarse de tipos en categóricas
data_new = ensure_object(data_new, CATEGORICAL_VARS)

# Detectar y dropear filas con NaN inesperados en features NO previstas
expected_na_vars = set(NUMERICAL_VARS_WITH_NA + CATEGORICAL_VARS_WITH_NA_MISSING + CATEGORICAL_VARS_WITH_NA_FREQUENT)
for col in FEATURES:
    if data_new[col].isnull().any() and col not in expected_na_vars:
        print(f"Advertencia: NaN inesperados en {col}. Drop rows.")
        data_new = data_new[data_new[col].notnull()]

print(f"Scoring sobre {data_new.shape[0]} filas limpias de datos nuevos...")
y_pred_new = price_pipe.predict(data_new)
y_pred_new_orig = np.exp(y_pred_new)

plt.figure(figsize=(6,3))
plt.hist(y_pred_new_orig, bins=30, edgecolor='k', alpha=0.7)
plt.title('Histograma de predicciones sobre datos nuevos')
plt.xlabel('Predicción')
plt.show()

> **Nota:**
> - Se dropean filas porque el pipeline SOLO fue entrenado para imputar ciertas features; otras deben llegar sin NaN. Esto es crucial en producción real.

---

## ¡Listo!
Este pipeline es autocontenible y está listo para `joblib.load` y para deployarse en cualquier backend productivo (`pipeline.predict(new_data)`). Todos los procesos (preprocessing, transformación, predicción) ocurren en UN SOLO OBJETO.